### 3D instantaneous_w

In [2]:
import os
import h5py
import matplotlib as mpl 
import matplotlib.pyplot as plt 
import matplotlib.colors as colors
from matplotlib import cm
from matplotlib.ticker import MultipleLocator
import numpy as np
import xarray as xr
from mpl_toolkits.mplot3d import Axes3D  # 确保 3D 引擎加载

# ==================== 1. 基础物理参数与网格固定配置 ====================
H = 25
nz = 101
lx = 600  # m
ly = 600  # m
lz = H    # m
dx = 600 / 600
dy = 600 / 600
dz = H / nz

# 建立 1D 原始物理坐标 (单位: m)
x1d_full = np.arange(dx / 2, lx, dx)
y1d_full = np.arange(dy / 2, ly, dy)
z1d_full = np.linspace(-lz, 0, nz)

# 对所有 1D 坐标进行 H 归一化 (变为无量纲)
x1d_norm = x1d_full / H
y1d_norm = y1d_full / H
z1d_norm = z1d_full / H  # 范围变为 -1 到 0

# 截取四分之一区域 (归一化之后的范围截取)
lx_plot_norm = (lx / 2) / H  # 12
ly_plot_norm = (ly / 2) / H  # 12

x1d = x1d_norm[x1d_norm <= lx_plot_norm]
y1d = y1d_norm[y1d_norm <= ly_plot_norm]
z1d = z1d_norm  # 保持 -1 到 0

# 转换为 3D 绘图所需的归一化 2D 网格
X_xy, Y_xy = np.meshgrid(x1d, y1d)      # 用于 XY 平面
X_xz, Z_xz = np.meshgrid(x1d, z1d)      # 用于 XZ 平面
Z_yz, Y_yz = np.meshgrid(z1d, y1d)      # 用于 YZ 平面

# ==================== 2. 定义待处理的 NC 文件列表 ====================
file_paths = [
    "G:/Lat035 theta05 kH10/velocity_3h_Lat035_theta05_kH10_part1.nc",
    "G:/Lat07 theta2 kH10/velocity_3h_Lat07_theta2_kH10_part1.nc",
    "G:/LatNaN theta1 NaN/velocity_3h_LatNaN_theta1_NaN_part1.nc",
    "G:/Lat035 theta05 kH2_5/velocity_3h_Lat035_theta05_kH2\'5_part1.nc",
    "G:/Lat035 theta2 kH2_5/velocity_3h_Lat035_theta2_kH2\'5_part1.nc",
    "G:/Lat07 theta05 kH2_5/velocity_3h_Lat07_theta05_kH2\'5_part1.nc"
]

# 设置图片保存的根目录
output_dir = "C:/Users/Administrator/Desktop/instantaneous_w_new"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

title=["$(Lat, \chi, kH) = (0.35, 0.5, 10)$",
      "$(Lat, \chi, kH) = (0.7, 2, 10)$",
      "$(Lat, \chi, kH) = (\infty, 1, -)$",
      "$(Lat, \chi, kH) = (0.35, 0.5, 2.5)$",
      "$(Lat, \chi, kH) = (0.35, 2, 2.5)$",
      "$(Lat, \chi, kH) = (0.7, 0.5, 2.5)$"]

# ==================== 3. 循环批量绘图函数 ====================
for i,path in enumerate(file_paths[0:3]):
    if not os.path.exists(path):
        print(f"警告: 文件不存在，跳过 -> {path}")
        continue
        
    # 自动解析路径，提取文件夹名字作为图片文件名
    folder_name = os.path.basename(os.path.dirname(path))
    
    # 【核心修改：根据文件夹名称动态匹配 ustar】
    if "kH10" in folder_name:
        ustar = 0.006050926454703675
    elif "kH5" in folder_name:
        ustar = 0.008568206405586257
    else:
        # 对应 kH2_5、kH2'5 以及 NaN 况
        ustar = 0.01214
        
    print(f"正在处理: {folder_name} (匹配 ustar = {ustar}) ...")
    
    # 读取数据
    data = xr.open_dataset(path)
    w = data.w / ustar
    it = 0  # 提取第一个时刻
    
    # 提取三个切片的数据并进行四分之一裁剪
    w_xy = w[it, 95, :len(y1d), :len(x1d)].values
    w_xz = w[it, :, 0, :len(x1d)].values
    w_yz = w[it, :, :len(y1d), 0].values.T
    
    # 关闭数据集释放内存
    data.close()

    # 开始绘图
    plt.rcParams.update({
    'font.size': 18,         
    'axes.labelsize': 20,   
    'xtick.labelsize': 18,
    'axes.titlesize': 20,
    'ytick.labelsize': 18  
    # 'font.family': 'serif'    # 顺便统一字体衬线风格（选填）
    })
    
    fig = plt.figure(figsize=(12, 7), dpi=300)
    ax = fig.add_subplot(111, projection='3d')

    # 色彩与刻度配置
    levels_w = np.linspace(-3, 3, 21)
    cmap_w = cm.get_cmap("RdYlBu_r")
    norm = colors.Normalize(vmin=-3, vmax=3)

    # --- 将三个归一化后的平面投射到 3D 空间中 ---
    # 1. 顶部表面 (XY平面)
    ax.contourf(X_xy, Y_xy, w_xy, zdir='z', offset=0, levels=levels_w, cmap=cmap_w, alpha=0.9, extend='both')
    # 2. 后侧墙面 (XZ平面)
    ax.contourf(X_xz, w_xz, Z_xz, zdir='y', offset=ly_plot_norm, levels=levels_w, cmap=cmap_w, alpha=0.9, extend='both')
    # 3. 左侧墙面 (YZ平面)
    ax.contourf(w_yz, Y_yz, Z_yz, zdir='x', offset=0, levels=levels_w, cmap=cmap_w, alpha=0.9, extend='both')

    # --- 3D 空间格式美化 ---
    ax.set_xlim(0, lx_plot_norm)
    ax.set_ylim(0, ly_plot_norm)
    ax.set_zlim(-1, 0)

    # 修改坐标轴标签
    ax.set_xlabel('x/H', labelpad=15)
    ax.set_ylabel('y/H', labelpad=10)
    ax.set_zlabel('z/H', labelpad=15)

    ax.set_title(f"{title[i]}", fontsize=28, pad=2, fontweight='bold')

    # 刻度间隔设置
    xspacing_norm = 4.0
    yspacing_norm = 4.0
    zspacing_norm = 0.2
    ax.xaxis.set_major_locator(MultipleLocator(xspacing_norm))
    ax.yaxis.set_major_locator(MultipleLocator(yspacing_norm))
    ax.zaxis.set_major_locator(MultipleLocator(zspacing_norm))

    # 调整视觉比例与显示视角
    ax.set_box_aspect((1, 1, 0.5)) 
    ax.view_init(elev=35, azim=155)

    # --- 添加全局色标 (两头延伸) ---
    sm = plt.cm.ScalarMappable(cmap=cmap_w, norm=norm)
    sm.set_array([])
    cbar_ax = plt.axes([0.75, 0.55, 0.025, 0.25])
    cbar = fig.colorbar(sm, cax=cbar_ax, ticks=levels_w[::4], extend='both')
    cbar.ax.set_title("$w/u_{*s}$", pad=10)

    # --- 保存并关闭画布 ---
    # 替换文件名中的非法字符（如单引号 ' 等），避免部分操作系统下保存失败
    safe_folder_name = folder_name.replace(' ', '_').replace("'", '_')
    save_path = os.path.join(output_dir, f"3D_cube_{safe_folder_name}.png")
    
    fig.savefig(save_path, bbox_inches='tight')
    plt.close(fig)  # 释放内存
    
print("所有图片批量生成完毕！")

正在处理: LatNaN theta1 NaN (匹配 ustar = 0.01214) ...


C:\Users\Administrator\AppData\Local\Temp\ipykernel_14068\2651948202.py:88: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  data = xr.open_dataset(path)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_14068\2651948202.py:115: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap_

所有图片批量生成完毕！
